# Day 7 (Sunday): EDA Mini-Project — Titanic Dataset
## Week 1 — Python for ML & Math Refresh

**Time:** 3 hours  
**Goal:** Apply EVERYTHING from this week in a real exploratory data analysis. This is the exact workflow you'll follow before building any ML model.

> 💡 **Project structure:** This follows the standard EDA workflow used in industry and Kaggle competitions. By the end, you'll have a template you can reuse for any dataset.

---

## 1. Project Setup & Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

# ── Load Titanic dataset ──
# Download from: https://www.kaggle.com/c/titanic/data
# Or use seaborn's built-in version:
df = sns.load_dataset('titanic')

print(f"Dataset shape: {df.shape}")
print(f"\nFirst 5 rows:")
df.head()

In [ ]:
# ── Initial inspection ──
print("=== INFO ===")
df.info()
print("\n=== DESCRIBE ===")
print(df.describe())
print("\n=== NULL COUNTS ===")
print(df.isnull().sum().sort_values(ascending=False))
print(f"\n=== NULL PERCENTAGES ===")
print((df.isnull().mean() * 100).round(1).sort_values(ascending=False))

## 2. Univariate Analysis — Understanding Each Variable

In [ ]:
# ── Target variable: Survived ──
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Count
survived_counts = df['survived'].value_counts()
axes[0].bar(['Did not survive', 'Survived'], survived_counts.values, 
           color=['#BF0000', '#548235'])
axes[0].set_title('Survival Count', fontweight='bold')
for i, v in enumerate(survived_counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# Percentage
axes[1].pie(survived_counts.values, labels=['Did not survive', 'Survived'],
           autopct='%1.1f%%', colors=['#BF0000', '#548235'], startangle=90)
axes[1].set_title('Survival Rate', fontweight='bold')

plt.tight_layout()
plt.savefig('survival_distribution.png', dpi=150)
plt.show()

print(f"Survival rate: {df['survived'].mean()*100:.1f}%")
# 💡 Imbalanced classes! Only ~38% survived. This affects model evaluation.

In [ ]:
# ── Numeric distributions ──
numeric_cols = ['age', 'fare', 'sibsp', 'parch']

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for i, col in enumerate(numeric_cols):
    ax = axes[i // 2, i % 2]
    df[col].dropna().hist(bins=30, ax=ax, color='#2E75B6', edgecolor='white')
    ax.axvline(df[col].mean(), color='red', linestyle='--', label=f'Mean: {df[col].mean():.1f}')
    ax.axvline(df[col].median(), color='green', linestyle='--', label=f'Median: {df[col].median():.1f}')
    ax.set_title(f'{col} Distribution', fontweight='bold')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('numeric_distributions.png', dpi=150)
plt.show()

# 💡 Age is roughly normal, Fare is heavily right-skewed
# 💡 Most passengers travelled alone (sibsp=0, parch=0)

In [ ]:
# ── Categorical distributions ──
cat_cols = ['sex', 'pclass', 'embarked', 'class']

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for i, col in enumerate(cat_cols):
    ax = axes[i // 2, i % 2]
    df[col].value_counts().plot(kind='bar', ax=ax, color='#2E75B6', edgecolor='white')
    ax.set_title(f'{col} Distribution', fontweight='bold')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('categorical_distributions.png', dpi=150)
plt.show()

## 3. Bivariate Analysis — What Predicts Survival?

In [ ]:
# ── Survival by categorical features ──
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# By sex
sns.barplot(x='sex', y='survived', data=df, ax=axes[0, 0], palette=['#2E75B6', '#ED7D31'])
axes[0, 0].set_title('Survival Rate by Sex', fontweight='bold')
axes[0, 0].set_ylabel('Survival Rate')

# By class
sns.barplot(x='pclass', y='survived', data=df, ax=axes[0, 1], palette='Blues_d')
axes[0, 1].set_title('Survival Rate by Class', fontweight='bold')

# By embarked
sns.barplot(x='embarked', y='survived', data=df, ax=axes[1, 0], palette='Set2')
axes[1, 0].set_title('Survival Rate by Embarkation Port', fontweight='bold')

# By sex AND class
sns.barplot(x='pclass', y='survived', hue='sex', data=df, ax=axes[1, 1], palette=['#2E75B6', '#ED7D31'])
axes[1, 1].set_title('Survival by Class & Sex', fontweight='bold')

plt.tight_layout()
plt.savefig('survival_by_category.png', dpi=150)
plt.show()

# Print survival rates
print("Survival rates:")
for col in ['sex', 'pclass', 'embarked']:
    print(f"\nBy {col}:")
    print(df.groupby(col)['survived'].mean().round(3))

In [ ]:
# ── Survival by age ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Age distribution by survival
for survived, color, label in [(0, '#BF0000', 'Did not survive'), (1, '#548235', 'Survived')]:
    subset = df[df['survived'] == survived]['age'].dropna()
    axes[0].hist(subset, bins=30, alpha=0.6, color=color, label=label, density=True)
axes[0].set_title('Age Distribution by Survival', fontweight='bold')
axes[0].set_xlabel('Age')
axes[0].legend()

# Box plot
sns.boxplot(x='survived', y='age', data=df, ax=axes[1], palette=['#BF0000', '#548235'])
axes[1].set_xticklabels(['Did not survive', 'Survived'])
axes[1].set_title('Age by Survival', fontweight='bold')

plt.tight_layout()
plt.savefig('survival_by_age.png', dpi=150)
plt.show()

## 4. Correlation Analysis

In [ ]:
# ── Correlation heatmap ──
numeric_df = df[['survived', 'pclass', 'age', 'sibsp', 'parch', 'fare']].dropna()
corr = numeric_df.corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))  # upper triangle mask
sns.heatmap(corr, mask=mask, annot=True, cmap='RdBu_r', center=0, fmt='.2f',
            square=True, linewidths=1, ax=ax)
ax.set_title('Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('titanic_correlations.png', dpi=150)
plt.show()

# Key observations:
print("Key correlations with survival:")
print(corr['survived'].sort_values(ascending=False).round(3))

## 5. Missing Data Strategy

In [ ]:
# ── Analyse missing data patterns ──
missing = df.isnull().sum()
missing_pct = (df.isnull().mean() * 100).round(1)
missing_df = pd.DataFrame({'count': missing, 'percent': missing_pct})
missing_df = missing_df[missing_df['count'] > 0].sort_values('percent', ascending=False)
print("Missing data:")
print(missing_df)

print("\n=== IMPUTATION STRATEGY ===")
print(f"age:      {missing_df.loc['age', 'percent']}% missing → impute with MEDIAN by class")
print(f"deck:     {missing_df.loc['deck', 'percent']}% missing → too much missing, DROP column")
print(f"embarked: small % missing → impute with MODE")
print(f"embark_town: same as embarked")

# ── Apply imputation ──
df_clean = df.copy()

# Age: median by class (more accurate than overall median)
df_clean['age'] = df_clean.groupby('pclass')['age'].transform(
    lambda x: x.fillna(x.median())
)

# Embarked: mode
df_clean['embarked'] = df_clean['embarked'].fillna(df_clean['embarked'].mode()[0])

# Drop deck (too many missing)
df_clean = df_clean.drop(columns=['deck'])

print(f"\nRemaining nulls: {df_clean.isnull().sum().sum()}")

## 6. Feature Engineering Ideas

In [ ]:
# ── Create new features from existing data ──

# Family size
df_clean['family_size'] = df_clean['sibsp'] + df_clean['parch'] + 1
df_clean['is_alone'] = (df_clean['family_size'] == 1).astype(int)

# Age groups
df_clean['age_group'] = pd.cut(df_clean['age'], bins=[0, 12, 18, 35, 60, 100],
                                labels=['Child', 'Teen', 'Young Adult', 'Adult', 'Senior'])

# Fare per person
df_clean['fare_per_person'] = df_clean['fare'] / df_clean['family_size']

# Title from name (powerful feature!)
df_clean['title'] = df_clean['who']  # simplified — in real project, extract from name

# ── Visualise new features ──
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

sns.barplot(x='family_size', y='survived', data=df_clean, ax=axes[0], palette='Blues_d')
axes[0].set_title('Survival by Family Size', fontweight='bold')

sns.barplot(x='age_group', y='survived', data=df_clean, ax=axes[1], palette='Set2')
axes[1].set_title('Survival by Age Group', fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)

sns.barplot(x='is_alone', y='survived', data=df_clean, ax=axes[2], palette=['#ED7D31', '#548235'])
axes[2].set_xticklabels(['With Family', 'Alone'])
axes[2].set_title('Survival: Alone vs Family', fontweight='bold')

plt.tight_layout()
plt.savefig('engineered_features.png', dpi=150)
plt.show()

## 7. EDA Summary & Key Findings

In [ ]:
# ── Summary Report ──
print("=" * 60)
print("TITANIC EDA — KEY FINDINGS")
print("=" * 60)

summary_text = (
    "1. SURVIVAL RATE: ~38%% survived (imbalanced target)\n\n"
    "2. STRONGEST PREDICTORS:\n"
    "   - Sex: Women survived at 74%% vs men at 19%%\n"
    "   - Class: 1st class 63%%, 2nd 47%%, 3rd 24%%\n"
    "   - Age: Children had higher survival rates\n"
    "   - Fare: Higher fare = higher survival (correlated with class)\n\n"
    "3. FEATURE ENGINEERING:\n"
    "   - Family size of 2-4 had best survival rates\n"
    "   - Solo travellers had lower survival\n"
    "   - Age groups show clear differentiation\n\n"
    "4. MISSING DATA:\n"
    "   - Age: 20%% missing, imputed by class median\n"
    "   - Deck: 77%% missing, dropped\n"
    "   - Embarked: <1%% missing, imputed with mode\n\n"
    "5. DATA ISSUES TO NOTE:\n"
    "   - Fare is heavily right-skewed, consider log transform\n"
    "   - Class 3 is overrepresented, stratified sampling needed\n"
    "   - Age and Fare have outliers, consider capping\n\n"
    "6. NEXT STEPS (Week 4):\n"
    "   - Encode categorical variables\n"
    "   - Scale numerical features\n"
    "   - Build baseline model (Logistic Regression)\n"
    "   - Compare with tree-based models"
)
print(summary_text)

## 8. EDA Template Checklist

Use this for every new dataset:

| Step | Action | Done? |
|------|--------|-------|
| 1 | Load & inspect shape, dtypes, head/tail | ✅ |
| 2 | Check missing values (count + %) | ✅ |
| 3 | Describe numeric features (stats) | ✅ |
| 4 | Plot univariate distributions | ✅ |
| 5 | Analyse target variable balance | ✅ |
| 6 | Bivariate: target vs each feature | ✅ |
| 7 | Correlation matrix heatmap | ✅ |
| 8 | Plan missing data strategy | ✅ |
| 9 | Feature engineering ideas | ✅ |
| 10 | Document findings | ✅ |

---

## ✅ WEEK 1 COMPLETE!

**What you now know:**
- NumPy: arrays, broadcasting, vectorised operations
- Pandas: DataFrames, groupby, merge, pivot
- Matplotlib/Seaborn: publication-quality plots
- Linear algebra: vectors, matrices, dot products, eigenvalues
- Statistics: distributions, Bayes, hypothesis testing, correlation
- EDA workflow: a repeatable template for any dataset

**Next week:** Core ML concepts — supervised vs unsupervised, bias-variance tradeoff, evaluation metrics, and your first ML models from scratch.